# Vector Quantization, ANN, and Re-ranking in Production RAG Systems

## 1. Why Quantization Exists

Embedding vectors consume a large amount of memory.

Example:

* Embedding dimension = 1536
* Data type = Float32
* Float32 size = 4 bytes

Memory per vector:

```text
1536 × 4 = 6144 bytes ≈ 6 KB
```

For 100 million vectors:

```text
100,000,000 × 6144 bytes
≈ 614 GB
```

This quickly becomes expensive.

Quantization reduces vector storage and improves search speed.

---

# 2. Types of Quantization

## 2.1 Scalar Quantization (Int8)

Converts Float32 values into Int8 values.

Example:

```text
Float32:
[0.75, -0.31, 0.91]

Int8:
[95, -40, 116]
```

### Memory Reduction

```text
Float32 = 4 bytes/value
Int8    = 1 byte/value
```

Compression:

```text
4× reduction
```

### Advantages

* Minimal accuracy loss
* Fast distance computation
* Easy to deploy

### Typical Use Case

```text
HNSW + Int8
```

for enterprise RAG systems.

---

## 2.2 Binary Quantization

Convert each dimension to:

```text
1 if value > 0
0 otherwise
```

Example:

```text
[0.7, -0.3, 1.2, -0.8]

↓

[1, 0, 1, 0]
```

### Memory Reduction

```text
32 bits → 1 bit
```

Compression:

```text
32× reduction
```

### Search Metric

Uses:

```text
Hamming Distance
```

instead of cosine similarity.

### Advantages

* Extremely fast
* Very low memory usage

### Disadvantages

* Significant information loss
* Lower recall

### Typical Use Case

```text
Billion-scale retrieval systems
```

---

## 2.3 Product Quantization (PQ)

PQ compresses vectors by representing groups of dimensions with centroid IDs.

Instead of storing:

```text
[0.12, 0.87, -0.44, ...]
```

Store:

```text
[23, 5, 91, 14, ...]
```

where each number references a learned centroid.

### Compression

Common compression ratios:

```text
16× to 64×
```

or even higher.

### Advantages

* Massive storage savings
* Very memory efficient

### Disadvantages

* More accuracy loss than Int8
* Requires approximation during distance calculations

### Typical Use Case

```text
100M+ vectors
```

---

# 3. ANN and Quantization Solve Different Problems

Many people think ANN and quantization are alternatives.

They are not.

They solve different problems.

---

## ANN Problem

Given:

```text
100M vectors
```

How do we avoid comparing against all 100M vectors?

Solution:

```text
HNSW
IVF
DiskANN
```

ANN reduces:

```text
Number of vectors searched
```

---

## Quantization Problem

Given:

```text
Each vector is large
```

How do we reduce memory and storage?

Solution:

```text
Int8
PQ
Binary
```

Quantization reduces:

```text
Size of each vector
```

---

## Combined Architecture

```text
Vectors
   ↓
Quantization
   ↓
ANN Index
   ↓
Search
```

Most production systems use:

```text
ANN + Quantization
```

together.

---

# 4. Re-ranking Architecture

Production systems often use a two-stage retrieval pipeline.

---

## Stage 1: Fast Candidate Retrieval

Search compressed vectors.

Example:

```text
Query
  ↓
PQ / Int8 / Binary Search
  ↓
Top 1000 Candidates
```

Fast but approximate.

---

## Stage 2: Exact Re-scoring

Fetch original vectors.

```text
Top 1000 IDs
    ↓
Load Original Vectors
    ↓
Compute Exact Similarity
    ↓
Top 20
```

This recovers most lost accuracy.

---

# 5. Why Store Original Vectors?

Without originals:

```text
Search Quality = Quantized Quality
```

With originals:

```text
Quantized Search
        +
Float Re-scoring
```

Result:

```text
Near Float32 Accuracy
```

---

# 6. Does Storing Originals Defeat Compression?

Not necessarily.

There are three modes.

---

## Mode A: Int8 + Original Float32

Store:

```text
Float32
+
Int8
```

Purpose:

```text
Speed
```

not storage.

Most common.

---

## Mode B: Int8 Only

Store:

```text
Int8
```

Purpose:

```text
Storage reduction
```

Compression:

```text
4×
```

---

## Mode C: PQ Only

Store:

```text
PQ codes
```

Purpose:

```text
Maximum storage reduction
```

Compression:

```text
16× to 64×
```

or more.

---

# 7. Float16 for Re-ranking

Instead of storing original vectors as Float32:

```text
Float32 = 4 bytes
```

store:

```text
Float16 = 2 bytes
```

Compression:

```text
50% reduction
```

while maintaining almost identical cosine similarity.

---

## Example

1536-dimensional embedding

### Float32

```text
1536 × 4 = 6144 bytes
```

### Float16

```text
1536 × 2 = 3072 bytes
```

Storage reduced by half.

---

# 8. Recommended Large-Scale Architecture

```text
RAM
 ├── HNSW Graph
 ├── PQ / Int8 / Binary Vectors

SSD
 └── Float16 Original Vectors
```

Search:

```text
Query
   ↓
Quantized ANN Search
   ↓
Top 1000 Candidates
   ↓
Load Float16 Vectors
   ↓
Exact Similarity
   ↓
Top 50
   ↓
Cross Encoder
   ↓
Top 10
```

---

# 9. Accuracy Comparison

Without Re-ranking:

```text
Float32
   >
Int8
   >
PQ
   >
Binary
```

---

# 10. Storage Comparison

For a 1536-dimensional vector:

| Method  | Approx Storage |
| ------- | -------------- |
| Float32 | 6144 bytes     |
| Float16 | 3072 bytes     |
| Int8    | 1536 bytes     |
| PQ      | 64–256 bytes   |
| Binary  | 192 bytes      |

---

# 11. When to Use What

## Up to 1M vectors

```text
Float32
```

Simple and accurate.

---

## 1M–50M vectors

```text
HNSW + Int8
```

Best balance of accuracy and efficiency.

---

## 50M–500M vectors

```text
HNSW + PQ
```

Memory starts becoming expensive.

---

## Billion-Scale Systems

```text
DiskANN
+
PQ
+
Float16 Re-ranking
```

or

```text
Binary
+
Re-ranking
```

for extreme scale.

---

# Final Takeaway

ANN and Quantization are complementary:

ANN:

* Reduces how many vectors are searched.

Quantization:

* Reduces how much memory each vector consumes.

Modern retrieval systems typically use:

```text
ANN
+
PQ/Int8/Binary
+
Float16 Re-scoring
+
Cross-Encoder Re-ranking
```

to achieve high recall, low latency, and manageable infrastructure costs.
